#**NLU Assignment-2**

---



## **PROBLEM 2: CHARACTER-LEVEL NAME GENERATION USING RNN VARIANTS**

### **TASK-0: DATASET**

In [ ]:
import os

file_path = "TrainingNames.txt"
assert os.path.exists(file_path), "File not found. Upload TrainingNames.txt first."

# loading names and converting to lowercase for consistency
with open(file_path, "r", encoding="utf-8") as f:
    names = [line.strip().lower() for line in f if line.strip()]

print("File loaded successfully")
print("Total names:", len(names))

# preview a few names
print("\nSample names:")
print(names[:10])

# checking duplicates
unique_names = set(names)
print("\nUnique names:", len(unique_names))
print("Duplicates:", len(names) - len(unique_names))

# checking length statistics
lengths = [len(name) for name in names]
print("\nMin length:", min(lengths))
print("Max length:", max(lengths))
print("Avg length:", sum(lengths)/len(lengths))

File loaded successfully
Total names: 1000

Sample names:
['aadhya', 'aadhyaesh', 'aadhyait', 'aadhyananda', 'aadhyauma', 'aarav', 'aaravendra', 'aaravsir', 'aaravvati', 'aarti']

Unique names: 1000
Duplicates: 0

Min length: 2
Max length: 13
Avg length: 7.759


### **TASK-1: MODEL IMPLEMENTATION**

**COMMON SETUP FOR ALL MODELS**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# splitting dataset into train, validation and test
train_size = int(0.8 * len(names))
val_size = int(0.1 * len(names))
test_size = len(names) - train_size - val_size

dataset = names
train_split, val_split, test_split = random_split(dataset, [train_size, val_size, test_size])

train_names = [dataset[i] for i in train_split.indices]
val_names = [dataset[i] for i in val_split.indices]
test_names = [dataset[i] for i in test_split.indices]

print(f"Train: {len(train_names)}, Val: {len(val_names)}, Test: {len(test_names)}")

# building character vocabulary
chars = sorted(list(set("".join(names))))
stoi = {ch:i+1 for i,ch in enumerate(chars)}
stoi["<pad>"] = 0
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(stoi)
print("Vocab size:", vocab_size)

# dataset class that creates input-target pairs
class NameDataset(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        x = torch.tensor([stoi[c] for c in name[:-1]])
        y = torch.tensor([stoi[c] for c in name[1:]])
        return x, y

# padding sequences so batch sizes are consistent
def collate_fn(batch):
    xs, ys = zip(*batch)
    xs = pad_sequence(xs, batch_first=True, padding_value=0)
    ys = pad_sequence(ys, batch_first=True, padding_value=0)
    return xs.to(device), ys.to(device)

train_loader = DataLoader(NameDataset(train_names), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(NameDataset(val_names), batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(NameDataset(test_names), batch_size=32, shuffle=False, collate_fn=collate_fn)

# training loop used by all models
def train_model(model, train_loader, val_loader, epochs, lr):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            optimizer.zero_grad()
            out = model(x)

            out = out.view(-1, vocab_size)
            y = y.view(-1)

            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch+1} | Loss: {total_loss:.3f} | Val Acc: {val_acc:.4f}")

# evaluation based on character-level accuracy
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            preds = out.argmax(dim=-1)

            mask = y != 0
            correct += (preds[mask] == y[mask]).sum().item()
            total += mask.sum().item()

    return correct / total if total > 0 else 0

Train: 800, Val: 100, Test: 100
Vocab size: 25


**MODEL 1: VANILLA RNN**

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers  # kept for consistency, but we use single-layer manually

        # embedding converts character indices into dense vectors
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)

        # defining weight matrices for RNN recurrence
        self.Wxh = nn.Linear(hidden_size, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size)

        # output layer maps hidden state to vocabulary logits
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        for t in range(seq_len):
            xt = self.embedding(x[:, t])  # current character embedding

            # update rule
            h = torch.tanh(self.Wxh(xt) + self.Whh(h))

            # computing output for current timestep
            out = self.fc(h)
            outputs.append(out.unsqueeze(1))

        # concatenating outputs across all timesteps
        return torch.cat(outputs, dim=1)


# hyperparameters
hidden_size = 128
num_layers = 1
lr = 0.003
epochs = 5

model_rnn = VanillaRNN(vocab_size, hidden_size, num_layers)

# counting trainable parameters
params = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)

print("Vanilla RNN\n")
print(f"Hidden size: {hidden_size}")
print(f"Number of layers: {num_layers}")
print(f"Learning rate: {lr}")
print(f"Trainable parameters: {params}")

# training the model
train_model(model_rnn, train_loader, val_loader, epochs, lr)

# evaluating on test set
test_acc = evaluate(model_rnn, test_loader)
print("\nTest Accuracy:", test_acc)

Vanilla RNN

Hidden size: 128
Number of layers: 1
Learning rate: 0.003
Trainable parameters: 39449
Epoch 1 | Loss: 57.894 | Val Acc: 0.3957
Epoch 2 | Loss: 45.795 | Val Acc: 0.4586
Epoch 3 | Loss: 40.396 | Val Acc: 0.5257
Epoch 4 | Loss: 36.510 | Val Acc: 0.5629
Epoch 5 | Loss: 33.536 | Val Acc: 0.5900

Test Accuracy: 0.5670553935860059


**MODEL 2: BLSTM**

In [ ]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        # defining LSTM gates
        self.Wf = nn.Linear(input_size + hidden_size, hidden_size)  # forget gate
        self.Wi = nn.Linear(input_size + hidden_size, hidden_size)  # input gate
        self.Wo = nn.Linear(input_size + hidden_size, hidden_size)  # output gate
        self.Wc = nn.Linear(input_size + hidden_size, hidden_size)  # candidate cell

    def forward(self, x, h, c):
        # combining input and previous hidden state
        combined = torch.cat([x, h], dim=1)

        # computing gates
        f = torch.sigmoid(self.Wf(combined))
        i = torch.sigmoid(self.Wi(combined))
        o = torch.sigmoid(self.Wo(combined))
        c_tilde = torch.tanh(self.Wc(combined))

        # updating cell state
        c = f * c + i * c_tilde

        # updating hidden state
        h = o * torch.tanh(c)

        return h, c


class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers  # kept for consistency

        # embedding layer
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)

        # forward and backward LSTM cells
        self.lstm_f = CustomLSTMCell(hidden_size, hidden_size)
        self.lstm_b = CustomLSTMCell(hidden_size, hidden_size)

        # output layer combines both directions
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        x = self.embedding(x)

        # forward direction
        hf = torch.zeros(batch_size, self.hidden_size).to(x.device)
        cf = torch.zeros(batch_size, self.hidden_size).to(x.device)
        forward_outputs = []

        for t in range(seq_len):
            hf, cf = self.lstm_f(x[:, t], hf, cf)
            forward_outputs.append(hf)

        # backward direction
        hb = torch.zeros(batch_size, self.hidden_size).to(x.device)
        cb = torch.zeros(batch_size, self.hidden_size).to(x.device)
        backward_outputs = []

        for t in reversed(range(seq_len)):
            hb, cb = self.lstm_b(x[:, t], hb, cb)
            backward_outputs.insert(0, hb)

        # combining forward and backward outputs
        outputs = []
        for f, b in zip(forward_outputs, backward_outputs):
            combined = torch.cat([f, b], dim=1)
            out = self.fc(combined)
            outputs.append(out.unsqueeze(1))

        return torch.cat(outputs, dim=1)


model_blstm = BLSTM(vocab_size, hidden_size, num_layers)

# counting parameters
params = sum(p.numel() for p in model_blstm.parameters() if p.requires_grad)

print("BLSTM\n")
print(f"Hidden size: {hidden_size}")
print(f"Number of layers: {num_layers}")
print(f"Learning rate: {lr}")
print(f"Trainable parameters: {params}")

# training
train_model(model_blstm, train_loader, val_loader, epochs, lr)

# testing
test_acc = evaluate(model_blstm, test_loader)
print("\nTest Accuracy:", test_acc)

BLSTM

Hidden size: 128
Number of layers: 1
Learning rate: 0.003
Trainable parameters: 272793
Epoch 1 | Loss: 47.503 | Val Acc: 0.7600
Epoch 2 | Loss: 15.764 | Val Acc: 0.9286
Epoch 3 | Loss: 5.926 | Val Acc: 0.9700
Epoch 4 | Loss: 2.983 | Val Acc: 0.9729
Epoch 5 | Loss: 1.869 | Val Acc: 0.9857

Test Accuracy: 0.978134110787172


**MODEL 3: RNN WITH BASIC ATTENTION**

In [ ]:
class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers  # kept for consistency

        # embedding layer converts characters to vectors
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)

        # defining RNN weights
        self.Wxh = nn.Linear(hidden_size, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size)

        # attention layer to score each timestep
        self.attn = nn.Linear(hidden_size, hidden_size)

        # output layer
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        hidden_states = []

        for t in range(seq_len):
            xt = self.embedding(x[:, t])

            # update rule
            h = torch.tanh(self.Wxh(xt) + self.Whh(h))

            hidden_states.append(h.unsqueeze(1))

        hidden_states = torch.cat(hidden_states, dim=1)  # (B, T, H)

        # computing attention scores
        attn_scores = torch.tanh(self.attn(hidden_states))

        # converting scores to weights across time
        attn_weights = torch.softmax(attn_scores, dim=1)

        # applying attention (element-wise weighting)
        context = attn_weights * hidden_states

        # predicting output at each timestep
        out = self.fc(context)

        return out

model_attn = AttentionRNN(vocab_size, hidden_size, num_layers)

# counting parameters
params = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)

print("Attention RNN\n")
print(f"Hidden size: {hidden_size}")
print(f"Number of layers: {num_layers}")
print(f"Learning rate: {lr}")
print(f"Trainable parameters: {params}")

# training
train_model(model_attn, train_loader, val_loader, epochs, lr)

# testing
test_acc = evaluate(model_attn, test_loader)
print("\nTest Accuracy:", test_acc)

Attention RNN

Hidden size: 128
Number of layers: 1
Learning rate: 0.003
Trainable parameters: 55961
Epoch 1 | Loss: 71.702 | Val Acc: 0.3557
Epoch 2 | Loss: 57.667 | Val Acc: 0.3600
Epoch 3 | Loss: 52.768 | Val Acc: 0.4029
Epoch 4 | Loss: 49.408 | Val Acc: 0.4343
Epoch 5 | Loss: 46.335 | Val Acc: 0.4457

Test Accuracy: 0.43731778425655976


### **TASK 2: QUANTITATIVE EVALUATION**

**NAME GENERATION**

In [ ]:
def generate_name(model, max_len=7, temperature=0.65, min_len=4):
    model.eval()

    # starting with a realistic prefix taken from training data
    start_name = random.choice(train_names)
    name = start_name[:random.randint(2,3)]

    for _ in range(max_len):

        x = torch.tensor([[stoi[c] for c in name]]).to(device)
        out = model(x)

        # applying temperature to control randomness of predictions
        logits = out[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        probs[0] = 0

        idx = torch.multinomial(probs, 1).item()
        ch = itos[idx]

        name += ch

        # allowing early stopping only after minimum length is reached
        if len(name) >= min_len and random.random() < 0.2:
            break

    # returning name with first letter capitalized
    return name.capitalize()

**NOVELTY + DIVERSITY**

In [ ]:
def evaluate_metrics(samples, train_names):

    samples_lower = [s.lower() for s in samples]
    train_set = set(train_names)

    # computing novelty as fraction of names not seen in training
    novel = [s for s in samples_lower if s not in train_set]
    novelty_rate = len(novel) / len(samples)

    # computing diversity as fraction of unique names
    diversity = len(set(samples_lower)) / len(samples)

    return novelty_rate, diversity

**EVALUATION**

In [ ]:
import os

# creating output directory if it does not exist
os.makedirs("output", exist_ok=True)

models = {
    "Vanilla_RNN": model_rnn,
    "BLSTM": model_blstm,
    "Attention_RNN": model_attn
}

results = {}

for name, model in models.items():

    # generating 100 names for each model
    samples = generate_samples(model, n=100)

    # calculating novelty rate and diversity score
    novelty, diversity = evaluate_metrics(samples, train_names)

    print(f"\n{name}")
    print(f"Novelty Rate: {novelty:.4f}")
    print(f"Diversity: {diversity:.4f}")

    file_path = f"output/{name}_samples.txt"

    with open(file_path, "w") as f:

        f.write(f"{name} Generated Names (Total: 100)\n\n")

        for i, sample in enumerate(samples, 1):
            f.write(f" {sample}\n")

    print(f"Saved file: {file_path}")

    results[name] = {
        "novelty": novelty,
        "diversity": diversity
    }


Sample outputs (VanillaRNN):
['Ayal', 'Tushansha', 'Sainrajati', 'Sures', 'Pallavitar', 'Nisha', 'Ayeshnande', 'Smje', 'Prayandee', 'Myrash']

Vanilla_RNN
Novelty Rate: 0.9000
Diversity: 1.0000
Saved file: output/Vanilla_RNN_samples.txt

Sample outputs (BLSTM):
['Gaulianali', 'Mehaya', 'Mani', 'Svir', 'Vita', 'Mahil', 'Pathilit', 'Rohankani', 'Navir', 'Smanalini']

BLSTM
Novelty Rate: 0.9700
Diversity: 0.9900
Saved file: output/BLSTM_samples.txt

Sample outputs (AttentionRNN):
['Zoyansharn', 'Nahinidhan', 'Ramanid', 'Sanavareem', 'Viva', 'Rajarav', 'Smyani', 'Nananita', 'Tanavitar', 'Surani']

Attention_RNN
Novelty Rate: 0.9800
Diversity: 0.9700
Saved file: output/Attention_RNN_samples.txt


In [ ]:
combined_content = ""
output_dir = "output"

for name, model in models.items():
    file_path = os.path.join(output_dir, f"{name}_samples.txt")
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            combined_content += f.read() + "\n\n"
    else:
        print(f"Warning: File not found for {name}: {file_path}")

combined_file_path = os.path.join(output_dir, "Generated_Names.txt")
with open(combined_file_path, "w", encoding="utf-8") as f:
    f.write(combined_content)

print(f"Combined generated names saved to: {combined_file_path}")

Combined generated names saved to: output/Generated_Names.txt
